# 05 — Assertion Views, ReadPolicy, and Precise Retract

This notebook demonstrates the v0.1 SDK assertion-ergonomics path after the `ReadPolicy` migration:

1. Write facts and keep the returned `asrt_id` values.
2. Use `ReadPolicy` as a call-site read/display policy with `policy=...`.
3. Use `EntitySnapshot -> FieldAssertions -> AssertionRecordSet` to select exact assertions.
4. Use `.where(...)`, `.at(...)`, `.version(...)`, `.by_id(...)`, and `.one()` before destructive actions.
5. Create a frozen assertion view with `fg.views.create(..., asrt_ids=...)` / `asrts=...`.
6. Read the frozen view back through `fg.assertions.by_ids(view.asrt_ids)`.
7. Keep frozen assertion views separate from `ReadPolicy` and old view-keyword call sites.

Every code cell asserts on the behavior it demonstrates.

## 0. Imports

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_src = cwd / "src"
if not repo_src.exists() and cwd.name == "examples":
    repo_src = cwd.parent / "src"
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

In [ ]:
from kernel.sdk import Entity, Field, Identity, ReadPolicy, SDKStore, SDKStoreError

## 1. Define a small schema

`User.user_id` is the primary identity. `User.locale` is a secondary coordinate dimension with a default. `name` is single-valued and `tag` is multi-valued.

In [ ]:
class User(Entity):
    user_id: str = Identity(primary_key=True)
    locale: str = Identity(default="en")
    name: str = Field(cardinality="single")
    tag: str = Field(cardinality="multi")

## 2. Write facts and keep assertion ids

`set(...)` and `add(...)` return persisted assertion ids. Those ids are the precise handles used later for frozen views and `retract(...)`.

In [ ]:
fg = SDKStore([User])
ref = fg.read.ref(User, user_id="u-1", locale="en")

ids = {
    "name_v1": fg.write.set(
        User.name,
        ref,
        "Alice",
        meta={
            "source": "seed",
            "trace_id": "import-001",
            "confidence": 0.82,
            "version": "name-v1",
            "valid_from": "2026-01-01T00:00:00Z",
            "valid_to": "2026-02-01T00:00:00Z",
        },
    ),
    "name_v2": fg.write.set(
        User.name,
        ref,
        "Alicia",
        meta={
            "source": "correction",
            "trace_id": "manual-001",
            "confidence": 0.95,
            "version": "name-v2",
            "valid_from": "2026-02-01T00:00:00Z",
        },
    ),
    "tag_vip": fg.write.add(
        User.tag,
        ref,
        "vip",
        meta={
            "source": "seed",
            "trace_id": "import-001",
            "confidence": 0.72,
            "version": "tag-v1",
            "valid_from": "2026-01-10T00:00:00Z",
            "valid_to": "2026-03-01T00:00:00Z",
        },
    ),
    "tag_legacy": fg.write.add(
        User.tag,
        ref,
        "legacy-no-valid-from",
        meta={"source": "legacy", "confidence": 0.2, "version": "tag-v0"},
    ),
}

assert set(ids) == {"name_v1", "name_v2", "tag_vip", "tag_legacy"}
assert all(isinstance(value, str) and value for value in ids.values())
ids

## 3. Use `ReadPolicy` at the read call site

`ReadPolicy` is a value object passed with `policy=...`. It is not stored in `fg.views` and has no named registry. `respect_revocations=True` means display/confidence aggregation skips claims that have an active retraction.

In [ ]:
policy = ReadPolicy(
    respect_revocations=True,
    confidence_strategy="max",
    prefer_source=None,
)

matches = fg.read.find(User, name="Alicia", policy=policy)
assert len(matches) == 1
assert matches[0].name == "Alicia"
assert matches[0].confidence == 0.95

without_policy = fg.read.find(User, name="Alicia")
assert len(without_policy) == 1
assert not hasattr(without_policy[0], "confidence")

## 4. Read a snapshot and inspect assertion records

Scalar fields are good for display; assertion collections are the path for audit, selection, view creation, and retract.

In [ ]:
snap = fg.read.get(User, user_id="u-1", locale="en")
assert snap is not None

assert snap.identity == {"user_id": "u-1", "locale": "en"}
assert snap.identity_available is True
assert isinstance(snap.name, str)

name_history = snap.field("name").history
same_path = snap.assertions.name.history

assert tuple(record.asrt_id for record in name_history) == tuple(
    record.asrt_id for record in same_path
)
assert hasattr(name_history, "where")
assert hasattr(name_history, "at")
assert hasattr(name_history, "version")
assert hasattr(name_history, "by_id")

## 5. Use `where`, `at`, `version`, and `by_id` as assertion filters

`.at(t)` is a business-time point filter over `valid_from` / `valid_to`, not an ingest-time filter. `valid_to` is exclusive.

In [ ]:
january_name = (
    snap.field("name")
    .history
    .where(source="seed", trace_id="import-001")
    .at("2026-01-15T00:00:00Z")
    .version("name-v1")
    .by_id(ids["name_v1"])
    .one()
)

assert january_name.value == "Alice"
assert january_name.meta.raw["valid_from"] == "2026-01-01T00:00:00Z"
assert january_name.meta.raw["valid_to"] == "2026-02-01T00:00:00Z"

february_name = snap.field("name").history.at("2026-02-01T00:00:00Z").version("name-v2").one()
assert february_name.value == "Alicia"

march_boundary = snap.field("tag").history.at("2026-03-01T00:00:00Z")
assert march_boundary.where(value="vip").all() == ()

missing_valid_from = snap.field("tag").history.by_id(ids["tag_legacy"])
assert missing_valid_from.at("2026-01-15T00:00:00Z").all() == ()

## 6. Create a frozen assertion view

A frozen assertion view names an immutable set of assertion ids. It can be created from ids or from objects exposing `.asrt_id`; only the ids become membership. There is no built-in `default` view, and the name `"default"` is not reserved.

In [ ]:
review_target = snap.field("name").history.where(value="Alice", source="seed").one()
review_tag = snap.field("tag").history.where(value="vip", source="seed").one()

view = fg.views.create("review_set", asrts=[review_target, review_tag])
default_named_view = fg.views.create("default", asrt_ids=[review_tag.asrt_id])

assert type(view).__name__ == "FrozenAssertionView"
assert view.name == "review_set"
assert isinstance(view.asrt_ids, frozenset)
assert view.asrt_ids == frozenset({ids["name_v1"], ids["tag_vip"]})
assert fg.views.get("review_set") is view
assert fg.views.get("default") is default_named_view
assert default_named_view.asrt_ids == frozenset({ids["tag_vip"]})

## 7. Read a frozen view back through `fg.assertions`

Frozen views are membership sets. To inspect the actual assertion records, use `fg.assertions.by_ids(view.asrt_ids)`.

In [ ]:
records = fg.assertions.by_ids(view.asrt_ids)

assert {record.asrt_id for record in records} == {ids["name_v1"], ids["tag_vip"]}
assert records.where(value="Alice").one().asrt_id == ids["name_v1"]
assert records.where(value="vip").one().asrt_id == ids["tag_vip"]
assert fg.assertions.by_id(ids["name_v1"]).value == "Alice"
assert fg.assertions.by_id("missing-asrt") is None

## 8. Keep frozen views separate from `ReadPolicy`

The old keyword named `view` is no longer a read/run call-site option, and a `FrozenAssertionView` is not a `ReadPolicy`. Use `fg.assertions.by_ids(...)` for record-level view readback; use `policy=ReadPolicy(...)` for read-time display/confidence aggregation.

In [ ]:
try:
    fg.read.find(User, **{"view": "review_set"})
except SDKStoreError as exc:
    view_message = str(exc)
else:
    raise AssertionError("the old view keyword must be rejected on read.find")

assert "view" in view_message
assert "policy=ReadPolicy" in view_message

try:
    fg.read.find(User, policy=view)
except SDKStoreError as exc:
    policy_message = str(exc)
else:
    raise AssertionError("FrozenAssertionView must not be accepted as a ReadPolicy")

assert "fg.read.find" in policy_message
assert "ReadPolicy" in policy_message
assert "FrozenAssertionView" in policy_message

## 9. Precise retract uses `target.asrt_id`

Do not retract by value or by `records[0]`. Use the read-side selectors to prove there is exactly one target, then pass its `asrt_id` to `write.retract(...)`.

In [ ]:
target = (
    fg.assertions.by_ids(view.asrt_ids)
    .where(value="Alice", source="seed", trace_id="import-001")
    .one()
)

revoker_asrt_id = fg.write.retract(
    target.asrt_id,
    meta={"source": "manual-fix", "trace_id": "fix-001"},
)
assert isinstance(revoker_asrt_id, str) and revoker_asrt_id

post_retract = fg.assertions.by_id(target.asrt_id)
assert post_retract is not None
assert post_retract.is_revoked is True

snap_after = fg.read.get(User, user_id="u-1", locale="en")
assert snap_after is not None
assert snap_after.field("name").history.by_id(target.asrt_id).one().is_revoked is True

## 10. Summary

The professional workflow is:

```text
write returns asrt_id
  -> ReadPolicy controls call-site confidence/display aggregation
  -> snapshot exposes AssertionRecordSet
    -> where/at/version/by_id narrow the set
      -> one() proves exactly one target
        -> views can freeze target ids
          -> fg.assertions.by_ids(...) reads them back
            -> write.retract(target.asrt_id) performs the mutation
```

This keeps assertion selection on the read side, read/display policy on the call site, and mutation on the write side.